# Resampling Strategy Experiment Results

## Objective

Evaluate the impact of different **class imbalance handling techniques** on model performance, including **Baseline (no resampling)**, **Class Weight**, **Random Under Sampling (RUS)**, and **Random Over Sampling (ROS)**.

## Top 10 Results

| Rank | Strategy | Best Model | CV AP | Train AP | Validation AP | Precision | Recall | F1 Score | Accuracy |
|-----:|----------|------------|------:|---------:|--------------:|----------:|-------:|---------:|---------:|
| 1 | Random Over Sampling | Gradient Boost | 0.105238 | 0.117420 | **0.086402** | 0.1155 | 0.7259 | 0.0627 | 0.8090 |
| 2 | Baseline | Logistic Regression (Elastic Net) | 0.107269 | 0.086275 | 0.086035 | 0.0000 | 0.0000 | 0.0000 | 0.9828 |
| 3 | Baseline | Logistic Regression (Lasso) | 0.105216 | 0.086823 | 0.085809 | 0.0000 | 0.0000 | 0.0000 | 0.9828 |
| 4 | Baseline | Logistic Regression (Base) | 0.104581 | 0.087257 | 0.085801 | 0.0000 | 0.0000 | 0.0000 | 0.9828 |
| 5 | Baseline | Logistic Regression (Ridge) | 0.107102 | 0.086486 | 0.085792 | 0.0000 | 0.0000 | 0.0000 | 0.9828 |
| 6 | Random Under Sampling | Logistic Regression (Lasso) | **0.108216** | 0.081576 | 0.083724 | 0.1140 | 0.7037 | 0.0620 | 0.8120 |
| 7 | Class Weight | Logistic Regression (Ridge) | 0.104874 | 0.085401 | 0.082897 | 0.1154 | 0.7185 | 0.0627 | 0.8108 |
| 8 | Class Weight | Logistic Regression (Elastic Net) | 0.104725 | 0.085411 | 0.082889 | 0.1154 | 0.7185 | 0.0627 | 0.8108 |
| 9 | Random Under Sampling | Logistic Regression (Elastic Net) | 0.110001 | 0.081139 | 0.082854 | 0.1151 | 0.7259 | 0.0625 | 0.8082 |
| 10 | Class Weight | Logistic Regression (Base) | 0.105517 | 0.085354 | 0.082523 | 0.1163 | 0.7259 | 0.0632 | 0.8104 |

## Conclusion

- **Random Over Sampling (ROS)** combined with **Gradient Boosting** achieved the highest **validation Average Precision (0.086402)** among all imbalance handling strategies.
- Although **Random Under Sampling** produced the highest cross-validation Average Precision, its validation performance was lower than both the baseline and Random Over Sampling, indicating reduced generalization.
- Applying **Class Weight** substantially increased recall compared with the baseline but did not improve validation Average Precision.
- The baseline Logistic Regression models remained highly competitive, suggesting that the original preprocessing pipeline already captured much of the predictive signal.
- Therefore, the final model adopts **Random Over Sampling (ROS)** as the imbalance handling strategy, as it provides the best validation performance while substantially improving minority class detection.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [3]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
SAVE_RESULT = "results/resampling_optimization_results.csv"
RANDOM_STATE=42

TARGET_TRANSFORM_COLS = ["Destination"]
LOG_TRANSFORM_COLS= ["Duration", "Net Sales"]

In [4]:
from src.utils import benchmark_models

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer

from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

In [5]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [6]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [8]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [9]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

numeric_log_pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])

In [10]:
preprocessor = ColumnTransformer(
    [
       
        ("numeric_pipeline", numeric_pipeline, [c for c in NUMERIC_COLS if c not in LOG_TRANSFORM_COLS]),
        ("numeric_log_pipeline", numeric_log_pipeline, LOG_TRANSFORM_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, [c for c in CATEGORICAL_COLS if c not in TARGET_TRANSFORM_COLS]),
        ("categorical_target_pipeline", categorical_target_pipeline, TARGET_TRANSFORM_COLS),

    ],
    remainder="drop"
)

In [11]:
pipeline = ImbPipeline(
    [
        ("preprocessor", preprocessor),
        ("resampler", "passthrough"), 
        ("classifier", LogisticRegression())
    ]
)

In [12]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RANDOM_STATE)
ab = AdaBoostClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [13]:
logreg_base_cw = LogisticRegression(
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

logreg_lasso_cw = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

logreg_ridge_cw = LogisticRegression(
    penalty="l2",
    solver="saga",
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

logreg_elasticnet_cw = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.5,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

dt_cw = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

rf_cw = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

xgb_cw = XGBClassifier(
    random_state=RANDOM_STATE,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)


In [14]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "DecisionTree": dt,
    "KNearestNeigbor": knn,
    "RandomForest": rf,
    "AdaBoost": ab,
    "GradientBoost": gb,
    "XGBoost": xgb
}

models_cw = {
    "LogisticRegressionBase": logreg_base_cw,
    "LogisticRegressionLasso": logreg_lasso_cw,
    "LogisticRegressionRidge": logreg_ridge_cw,
    "LogisticRegressionElasticNet": logreg_elasticnet_cw,
    "DecisionTree": dt_cw,
    "KNearestNeigbor": knn,          # KNN doesn't support class_weight
    "RandomForest": rf_cw,
    "AdaBoost": ab,                  # Doesn't support class_weight
    "GradientBoost": gb,             # Doesn't support class_weight
    "XGBoost": xgb_cw
}

ros = RandomOverSampler(random_state=RANDOM_STATE)
rus = RandomUnderSampler(random_state=RANDOM_STATE)
smote = SMOTE(random_state=RANDOM_STATE)

strategies = {
    "Baseline": ("passthrough", models),
    "ClassWeight": ("passthrough", models_cw),
    "RandomOverSampler": (ros, models),
    "RandomUnderSampler": (rus, models),
    "SMOTE": (smote, models),
}

results = pd.DataFrame()

for strategy_name, (resampler, model_dict) in strategies.items():
    pipeline.set_params(resampler=resampler)

    result = benchmark_models(
        pipeline=pipeline,
        list_model=model_dict,
        x_train=x_train,
        y_train=y_train,
        x_test=x_test,
        y_test=y_test,
        cv=5,
        postfix=strategy_name
    )

    results = pd.concat([results, result], ignore_index=True)

In [15]:
results = results.sort_values("mean_ap_validate_score", ascending=False)
results[0:10]

,name,ap_test_score,mean_ap_train_score,mean_ap_validate_score,f1_test_score,recall_test_score,precision_test_score,accuracy_test_score
20,GradientBoost_RandomOverSampler,0.105238,0.117420,0.086402,0.115498,0.725926,0.062740,0.808984
0,LogisticRegressionElasticNet_Baseline,0.107269,0.086275,0.086035,0.000000,0.000000,0.000000,0.982820
1,LogisticRegressionLasso_Baseline,0.105216,0.086823,0.085809,0.000000,0.000000,0.000000,0.982820
2,LogisticRegressionBase_Baseline,0.104581,0.087257,0.085801,0.000000,0.000000,0.000000,0.982820
3,LogisticRegressionRidge_Baseline,0.107102,0.086486,0.085792,0.000000,0.000000,0.000000,0.982820
30,LogisticRegressionLasso_RandomUnderSampler,0.108216,0.081576,0.083724,0.113977,0.703704,0.062010,0.812039
10,LogisticRegressionRidge_ClassWeight,0.104874,0.085401,0.082897,0.115407,0.718519,0.062743,0.810766
11,LogisticRegressionElasticNet_ClassWeight,0.104725,0.085411,0.082889,0.115407,0.718519,0.062743,0.810766
31,LogisticRegressionElasticNet_RandomUnderSampler,0.110001,0.081139,0.082854,0.115091,0.725926,0.062500,0.808221
12,LogisticRegressionBase_ClassWeight,0.105517,0.085354,0.082523,0.116251,0.725926,0.063185,0.810384


In [16]:
results.to_csv(SAVE_RESULT, index=False)